In [2]:
# =========================
# Standard library imports
# =========================
import sys
import os
from pathlib import Path
import math
import logging
import random
import subprocess
import warnings
import collections

from IPython.display import display

warnings.filterwarnings("ignore")


# =========================
# Numeric / stats
# =========================
import numpy as np
import pandas as pd
from scipy import stats
from scipy import __version__ as scipy_version


# =========================
# Plotting / visualization
# =========================
import matplotlib.pyplot as plt
import seaborn as sns


# =========================
# scikit-learn: split / CV / metrics
# =========================
from sklearn import model_selection, metrics
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
    KFold,
    GroupKFold,
    cross_validate,
)
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)


# =========================
# scikit-learn: models
# =========================
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import Ridge, Lasso
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import LocalOutlierFactor
from sklearn import __version__ as sklearn_version


# =========================
# Optimization
# =========================
import optuna


# =========================
# LightGBM
# =========================
try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
except Exception as e:
    raise RuntimeError(
        "LightGBM not installed. Please install it with `pip install lightgbm`."
    ) from e


# =========================
# XGBoost
# =========================
try:
    import xgboost as xgb
    from xgboost import XGBRegressor
except Exception as e:
    raise RuntimeError(
        "XGBoost not installed. Please install it with `pip install xgboost`."
    ) from e


# =========================
# RNA structure / user utilities
# =========================
import RNA
import sylib


# =========================
# Version info
# =========================
print(f"python    = {sys.version_info[0]}.{sys.version_info[1]}.{sys.version_info[2]}")
print(f"pandas    = {pd.__version__}")
print(f"numpy     = {np.__version__}")
print(f"scipy     = {scipy_version}")
print(f"sklearn   = {sklearn_version}")
print(f"optuna    = {optuna.__version__}")
print(f"lightgbm  = {lgb.__version__}")
print(f"xgboost   = {xgb.__version__}")
print(f"ViennaRNA = {RNA.__version__}")
print(f"sylib     = {sylib.__version__}")


# =========================
# Progress bar & logging
# =========================
progress_bar = sylib.utils.ProgressBar()

logging.root.handlers = []
stream_handler = logging.StreamHandler(sys.stderr)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler],
)

logger = logging.getLogger(__name__)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

python    = 3.11.15
pandas    = 2.3.3
numpy     = 2.4.6
scipy     = 1.17.1
sklearn   = 1.9.0
optuna    = 4.9.0
lightgbm  = 4.6.0
xgboost   = 3.2.0
ViennaRNA = 2.7.2
sylib     = 0.3.0.dev0+ae18bb2


In [3]:
# ============================================
# Benchmark settings
# ============================================

DATA_DIR = Path("../data/raw")

# prediction target
score_col_name = "PR_var"

# set to "gene_id" later if doing group-based split
group_col_name = None

# train/test split
test_data_ratio = 0.20

# reproducibility
random_state = 0

# CV
k_cv = 10

# parallelism
n_jobs = 10

# hyperparameter tuning
n_trials = 20

# species datasets and species-specific RNA folding temperature
species_config = {
    "AT21": {
        "file": DATA_DIR / "AT21.PR_var.dev_data.RNA.seq_data.tsv.gz",
        "rna_temp": 22,
    },
    "NB21": {
        "file": DATA_DIR / "NB21.PR_var.dev_data.RNA.seq_data.tsv.gz",
        "rna_temp": 22,
    },
    "OS21": {
        "file": DATA_DIR / "OS21.PR_var.dev_data.RNA.seq_data.tsv.gz",
        "rna_temp": 25,
    },
}

print("Species configuration:")
for species, cfg in species_config.items():
    print(f"{species}: file={cfg['file']}, rna_temp={cfg['rna_temp']}")

Species configuration:
AT21: file=../data/raw/AT21.PR_var.dev_data.RNA.seq_data.tsv.gz, rna_temp=22
NB21: file=../data/raw/NB21.PR_var.dev_data.RNA.seq_data.tsv.gz, rna_temp=22
OS21: file=../data/raw/OS21.PR_var.dev_data.RNA.seq_data.tsv.gz, rna_temp=25


In [4]:
# ============================================
# Check input files
# ============================================

for species, cfg in species_config.items():
    file_path = cfg["file"]
    print(species)
    print(" ", file_path)
    print(" exists:", file_path.exists())

AT21
  ../data/raw/AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
 exists: True
NB21
  ../data/raw/NB21.PR_var.dev_data.RNA.seq_data.tsv.gz
 exists: True
OS21
  ../data/raw/OS21.PR_var.dev_data.RNA.seq_data.tsv.gz
 exists: True


In [5]:
# ============================================
# Basic sequence helpers
# ============================================

def normalize_rna(seq):
    if pd.isna(seq):
        return ""
    return str(seq).upper().replace("T", "U")


def cat_mrna(utr5, cds, utr3):
    return normalize_rna(utr5) + normalize_rna(cds) + normalize_rna(utr3)

In [6]:
# ============================================
# Model registry: default benchmark models
# ============================================

def get_default_models(random_state=0, n_jobs=10):
    return {
        "PLS": PLSRegression(n_components=5),

        "Ridge": Ridge(alpha=1.0),

        "Lasso": Lasso(
            alpha=0.001,
            max_iter=10000,
            random_state=random_state,
        ),

        "MLP": MLPRegressor(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            alpha=0.0001,
            learning_rate_init=0.001,
            max_iter=500,
            random_state=random_state,
        ),

        "DecisionTree": DecisionTreeRegressor(
            random_state=random_state,
        ),

        "RandomForest": RandomForestRegressor(
            n_estimators=300,
            random_state=random_state,
            n_jobs=n_jobs,
        ),

        "XGBoost": XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=1.0,
            colsample_bytree=1.0,
            random_state=random_state,
            n_jobs=n_jobs,
        ),

        "LightGBM": LGBMRegressor(
            objective="regression",
            n_estimators=300,
            learning_rate=0.05,
            random_state=random_state,
            n_jobs=n_jobs,
            verbosity=-1,
        ),
    }


default_models = get_default_models(
    random_state=random_state,
    n_jobs=n_jobs,
)

list(default_models.keys())

['PLS',
 'Ridge',
 'Lasso',
 'MLP',
 'DecisionTree',
 'RandomForest',
 'XGBoost',
 'LightGBM']

In [10]:
# ============================================
# Load one species
# ============================================

def load_species_data(species, species_config):

    file_path = species_config[species]["file"]

    seq_df, metadata = sylib.fileio.load_df(
        str(file_path)
    )

    metadata.print_minimum_data(
        label=species,
        logger=logger,
        logging_level="info",
    )

    return seq_df, metadata

# Check Data Structure
seq_df, metadata = load_species_data(
    "AT21",
    species_config
)

print(seq_df.shape)

display(seq_df.head())

2026-06-14 18:05:21,516     INFO: AT21 |   name = AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:05:21,517     INFO: AT21 |    md5 = 463f63c03bb16bfb05e1f6ea025947b8
2026-06-14 18:05:21,517     INFO: AT21 | md5_gz = 7940f4f1aa886ba02c249422c074027e


(65, 11)


,var_id,trans_id,gene_id,usage,PR_var,5'UTR,CDS,3'UTR,n_5'UTR_introns,n_CDS_introns,n_3'UTR_introns
0,AT1G07770.2.2409543.2408264,AT1G07770.2,AT1G07770,0.043245,0.408580,CUCUUCUUCCUCUAAUUGCUUUUCUCCGUCACUGAGUACCUUGCCU...,AUGGUAAGAAUCAGUGUUCUUAACGAUGCUCUCAAGAGCAUGUACA...,AGUUUUAUUUUAUGGGGAAACAGAUUUUCAUUGAGUUAUUUUUAAC...,1,2,0
1,AT1G07770.2.2409543.2408265,AT1G07770.2,AT1G07770,0.035668,0.447437,CUCUUCUUCCUCUAAUUGCUUUUCUCCGUCACUGAGUACCUUGCCU...,AUGGUAAGAAUCAGUGUUCUUAACGAUGCUCUCAAGAGCAUGUACA...,AGUUUUAUUUUAUGGGGAAACAGAUUUUCAUUGAGUUAUUUUUAAC...,1,2,0
2,AT1G07770.3.2409543.2408265,AT1G07770.3,AT1G07770,0.037044,0.626719,CUCUUCUUCCUCUAAUUGCUUUUCUCCGUCACUGAGUACCUUGCCU...,AUGGUAAGAAUCAGUGUUCUUAACGAUGCUCUCAAGAGCAUGUACA...,AGUUUUAUUUUAUGGGGAAACAGAUUUUCAUUGAGUUAUUUUUAAC...,1,2,0
3,AT1G07820.1.2421940.2421219,AT1G07820.1,AT1G07820,0.111209,0.522178,GUCAAAUUCAAUUGAUCCUCUCUCCAAAUCAUCUUAAAAGUAUCUU...,AUGUCGGGAAGAGGAAAGGGAGGAAAAGGAUUGGGAAAGGGAGGAG...,UCCGAUUUGGGGGAUUAGGGUUUAUGCAAGUUUGGGGAUUUCUUCU...,0,0,0
4,AT1G07820.2.2421937.2421219,AT1G07820.2,AT1G07820,0.079275,0.490502,AAAUUCAAUUGAUCCUCUCUCCAAAUCAUCUUAAAAUCACAGAUCU...,AUGUCGGGAAGAGGAAAGGGAGGAAAAGGAUUGGGAAAGGGAGGAG...,UCCGAUUUGGGGGAUUAGGGUUUAUGCAAGUUUGGGGAUUUCUUCU...,1,0,0


In [15]:
# ============================================
# Feature Evaluation: Definition of Features
# Only benchmark features are included.
# ============================================

def eval_length(seq_list, region_name, logger=logger):
    return pd.DataFrame({
        f"{region_name}.Length": [
            len(seq) if isinstance(seq, str) else 0
            for seq in seq_list
        ]
    })


def eval_kmer_freq(seq_list, region_name, is_rna_seq=True, k=1, logger=logger):
    count_dict = sylib.sequtils.count_nuc_k_mer_of_seq_list(
        seq_list,
        is_rna_seq=is_rna_seq,
        k=k,
    )

    feat_df = pd.DataFrame({
        f"{region_name}.{key}-freq": count_dict[key]
        for key in count_dict
    })

    seq_len_sr = pd.Series([
        len(seq) if isinstance(seq, str) else 0
        for seq in seq_list
    ])

    seq_len_sr[seq_len_sr == 0] = 1

    for col_name in feat_df.columns:
        feat_df[col_name] /= seq_len_sr
        feat_df[col_name] *= 1000

    return feat_df


def eval_s_freq(seq_list, region_name, logger=logger):
    v = []
    for seq in seq_list:
        seq = seq if isinstance(seq, str) else ""
        if len(seq) == 0:
            v.append(0.0)
        else:
            v.append((seq.count("G") + seq.count("C")) / len(seq) * 1000)

    return pd.DataFrame({f"{region_name}.S-freq": v})


def eval_y_freq(seq_list, region_name, logger=logger):
    v = []
    for seq in seq_list:
        seq = seq if isinstance(seq, str) else ""
        if len(seq) == 0:
            v.append(0.0)
        else:
            v.append((seq.count("T") + seq.count("U") + seq.count("C")) / len(seq) * 1000)

    return pd.DataFrame({f"{region_name}.Y-freq": v})


def eval_k_freq(seq_list, region_name, logger=logger):
    v = []
    for seq in seq_list:
        seq = seq if isinstance(seq, str) else ""
        if len(seq) == 0:
            v.append(0.0)
        else:
            v.append((seq.count("G") + seq.count("T") + seq.count("U")) / len(seq) * 1000)

    return pd.DataFrame({f"{region_name}.K-freq": v})


def eval_n_uaug(utr5_seq_list, region_name, is_rna_seq=True, logger=logger):
    if is_rna_seq:
        return pd.DataFrame({
            f"{region_name}.uAUG": [
                seq.count("AUG") if isinstance(seq, str) else 0
                for seq in utr5_seq_list
            ]
        })
    else:
        return pd.DataFrame({
            f"{region_name}.uATG": [
                seq.count("ATG") if isinstance(seq, str) else 0
                for seq in utr5_seq_list
            ]
        })


def eval_n_uorf(utr5_seq_list, region_name, is_rna_seq=True, logger=logger):
    feat_df_dict = {
        f"{region_name}.uORF-NO": [],
        f"{region_name}.uORF-IO": [],
        f"{region_name}.uORF-OO": [],
    }

    if is_rna_seq:
        start_codon = "AUG"
        stop_codons = {"UAA", "UGA", "UAG"}
    else:
        start_codon = "ATG"
        stop_codons = {"TAA", "TGA", "TAG"}

    for seq in utr5_seq_list:
        seq = seq if isinstance(seq, str) else ""
        L = len(seq)

        n_no = 0
        n_io = 0
        n_oo = 0

        for i in range(L - 2):
            if seq[i:i+3] == start_codon:
                has_stop = False

                for j in range(i + 3, L - 2, 3):
                    if seq[j:j+3] in stop_codons:
                        has_stop = True
                        break

                if has_stop:
                    n_no += 1
                elif (L - i) % 3 == 0:
                    n_io += 1
                else:
                    n_oo += 1

        feat_df_dict[f"{region_name}.uORF-NO"].append(n_no)
        feat_df_dict[f"{region_name}.uORF-IO"].append(n_io)
        feat_df_dict[f"{region_name}.uORF-OO"].append(n_oo)

    return pd.DataFrame(feat_df_dict)


def eval_n_daug(utr3_seq_list, region_name, is_rna_seq=True, logger=logger):
    if is_rna_seq:
        return pd.DataFrame({
            f"{region_name}.dAUG": [
                seq.count("AUG") if isinstance(seq, str) else 0
                for seq in utr3_seq_list
            ]
        })
    else:
        return pd.DataFrame({
            f"{region_name}.dATG": [
                seq.count("ATG") if isinstance(seq, str) else 0
                for seq in utr3_seq_list
            ]
        })


def eval_n_dorf(utr3_seq_list, region_name, is_rna_seq=True, logger=logger):
    feat_df_dict = {
        f"{region_name}.dORF": [],
        f"{region_name}.dORF-NS": [],
    }

    if is_rna_seq:
        start_codon = "AUG"
        stop_codons = {"UAA", "UGA", "UAG"}
    else:
        start_codon = "ATG"
        stop_codons = {"TAA", "TGA", "TAG"}

    for seq in utr3_seq_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq + "AA"
        L = len(seq)

        n_no = 0
        n_ns = 0

        for i in range(L - 2):
            if seq[i:i+3] == start_codon:
                has_stop = False

                for j in range(i + 3, L - 2, 3):
                    if seq[j:j+3] in stop_codons:
                        has_stop = True
                        break

                if has_stop:
                    n_no += 1
                else:
                    n_ns += 1

        feat_df_dict[f"{region_name}.dORF"].append(n_no)
        feat_df_dict[f"{region_name}.dORF-NS"].append(n_ns)

    return pd.DataFrame(feat_df_dict)

# ============================================
# Feature Evaluation: CDS coding features
# Only benchmark non-MFE features are included here.
# MFE is precomputed separately by script.
# ============================================

def eval_codon_usage(cds_seq_list, region_name, is_rna_seq=True, should_remove_stop_codons=False, logger=logger):
    mer3_list = sylib.sequtils.get_k_mer_nuc_list(3, is_rna_seq=is_rna_seq)
    aa_codon_dict = sylib.sequtils.get_aa_codon_dict(
        is_rna_seq=is_rna_seq,
        code_type="3-letter",
    )

    n_codons_dict = {mer3: [0] * len(cds_seq_list) for mer3 in mer3_list}

    for i, cds_seq in enumerate(cds_seq_list):
        cds_seq = cds_seq if isinstance(cds_seq, str) else ""
        for j in range(0, len(cds_seq), 3):
            codon = cds_seq[j:j+3]
            if codon in n_codons_dict:
                n_codons_dict[codon][i] += 1

    codons_df = pd.DataFrame(n_codons_dict)
    usage_df_dict = {}

    for aa, codon_list in aa_codon_dict.items():
        aa_sr = pd.Series([0] * len(cds_seq_list))

        for codon in codon_list:
            aa_sr += codons_df[codon]

        aa_sr[aa_sr == 0] = 1

        for codon in codon_list:
            key = f"{region_name}.{aa}-{codon}"
            usage_df_dict[key] = (codons_df[codon] / aa_sr) * 1000

    if is_rna_seq:
        usage_df_dict.pop(f"{region_name}.Met-AUG", None)
        usage_df_dict.pop(f"{region_name}.Trp-UGG", None)

        if should_remove_stop_codons:
            for c in ("UAA", "UGA", "UAG"):
                usage_df_dict.pop(f"{region_name}.Ter-{c}", None)
    else:
        usage_df_dict.pop(f"{region_name}.Met-ATG", None)
        usage_df_dict.pop(f"{region_name}.Trp-TGG", None)

        if should_remove_stop_codons:
            for c in ("TAA", "TGA", "TAG"):
                usage_df_dict.pop(f"{region_name}.Ter-{c}", None)

    return pd.DataFrame(usage_df_dict)


def wobble_freq(cds_list, region_label="CDS"):
    alphabet = ["A", "U", "G", "C"]
    cols = {f"{region_label}.wobble_pct_{b}": [] for b in alphabet}

    for seq in cds_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq.upper().replace("T", "U")

        wobble_bases = [
            seq[i]
            for i in range(2, len(seq), 3)
            if i < len(seq)
        ]

        total = len(wobble_bases)

        for b in alphabet:
            cols[f"{region_label}.wobble_pct_{b}"].append(
                wobble_bases.count(b) / total if total > 0 else 0.0
            )

    return pd.DataFrame(cols)


CODON_TO_AA_RNA = {
    "UUU": "F", "UUC": "F", "UUA": "L", "UUG": "L",
    "UCU": "S", "UCC": "S", "UCA": "S", "UCG": "S",
    "UAU": "Y", "UAC": "Y", "UAA": "X", "UAG": "X",
    "UGU": "C", "UGC": "C", "UGA": "X", "UGG": "W",
    "CUU": "L", "CUC": "L", "CUA": "L", "CUG": "L",
    "CCU": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAU": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGU": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "AUU": "I", "AUC": "I", "AUA": "I", "AUG": "M",
    "ACU": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAU": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGU": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GUU": "V", "GUC": "V", "GUA": "V", "GUG": "V",
    "GCU": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAU": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGU": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}

AA_LIST = sorted(set(CODON_TO_AA_RNA.values()))


def aa_freq_from_cds(cds_list, region_label="CDS"):
    cols = {f"{region_label}.aa_pct_{aa}": [] for aa in AA_LIST}

    for seq in cds_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq.upper().replace("T", "U")

        counts = {aa: 0 for aa in AA_LIST}
        total = 0

        for i in range(0, len(seq) - 2, 3):
            codon = seq[i:i+3]
            aa = CODON_TO_AA_RNA.get(codon)

            if aa is None:
                continue

            counts[aa] += 1
            total += 1

        for aa in AA_LIST:
            cols[f"{region_label}.aa_pct_{aa}"].append(
                counts[aa] / total if total > 0 else 0.0
            )

    return pd.DataFrame(cols)


DICODONS_DNA = [
    "AGGCGA", "AGGCGG", "ATACGA", "ATACGG", "CGAATA",
    "CGACCG", "CGACGA", "CGACGG", "CGACTG", "CGAGCG",
    "CTCATA", "CTCCCG", "CTGATA", "CTGCCG", "CTGCGA",
    "CTGCTG", "CTTCTG", "GTACCG", "GTACGA", "GTGCGA",
]

DICODONS_RNA = [d.replace("T", "U") for d in DICODONS_DNA]


def dicodon_counts_rna(cds_list, region_label="CDS", normalized=False):
    cols = {f"{region_label}.dicodon_{d}": [] for d in DICODONS_RNA}

    for seq in cds_list:
        seq = seq if isinstance(seq, str) else ""
        seq = seq.upper().replace("T", "U")

        counts = {d: 0 for d in DICODONS_RNA}
        total = 0

        for i in range(0, len(seq) - 5, 3):
            dicodon = seq[i:i+6]

            if len(dicodon) == 6:
                total += 1

                if dicodon in counts:
                    counts[dicodon] += 1

        for d in DICODONS_RNA:
            value = counts[d] / total if (normalized and total > 0) else counts[d]
            cols[f"{region_label}.dicodon_{d}"].append(value)

    return pd.DataFrame(cols)

In [19]:
# ============================================
# Feature Evaluation: Execution of non-MFE features
# ============================================

def build_non_mfe_features(df):
    parts = []

    # 1. Region length
    parts += [
        eval_length(df["5'UTR"], "5'UTR"),
        eval_length(df["CDS"],   "CDS"),
        eval_length(df["3'UTR"], "3'UTR"),
    ]

    # 2. Region nucleotide / k-mer composition
    for k in [1, 2, 3]:
        parts += [
            eval_kmer_freq(df["5'UTR"], "5'UTR", is_rna_seq=True, k=k),
            eval_kmer_freq(df["CDS"],   "CDS",   is_rna_seq=True, k=k),
            eval_kmer_freq(df["3'UTR"], "3'UTR", is_rna_seq=True, k=k),
        ]

    # 3. Degenerate nucleotide class features
    parts += [
        eval_s_freq(df["5'UTR"], "5'UTR"),
        eval_y_freq(df["5'UTR"], "5'UTR"),
        eval_k_freq(df["5'UTR"], "5'UTR"),
        eval_s_freq(df["3'UTR"], "3'UTR"),
        eval_y_freq(df["3'UTR"], "3'UTR"),
        eval_k_freq(df["3'UTR"], "3'UTR"),
    ]

    # 4. Upstream/downstream ORF-related features
    parts += [
        eval_n_uaug(df["5'UTR"], "5'UTR", is_rna_seq=True),
        eval_n_uorf(df["5'UTR"], "5'UTR", is_rna_seq=True),
        eval_n_daug(df["3'UTR"], "3'UTR", is_rna_seq=True),
        eval_n_dorf(df["3'UTR"], "3'UTR", is_rna_seq=True),
    ]

    # 5. CDS coding features
    parts += [
        eval_codon_usage(df["CDS"], "CDS", is_rna_seq=True, should_remove_stop_codons=True),
        wobble_freq(df["CDS"], "CDS"),
        aa_freq_from_cds(df["CDS"], "CDS"),
        dicodon_counts_rna(df["CDS"], "CDS", normalized=True),
    ]

    x_feat_df = pd.concat(parts, axis=1)

    return x_feat_df

x_non_mfe_df = build_non_mfe_features(seq_df)

print(x_non_mfe_df.shape)
display(x_non_mfe_df.head())

# ============================================
# Check non-MFE feature matrix
# ============================================

print("shape:", x_non_mfe_df.shape)
print("missing values:", x_non_mfe_df.isna().sum().sum())
print("infinite values:", np.isinf(x_non_mfe_df.to_numpy()).sum())
print("duplicated columns:", x_non_mfe_df.columns.duplicated().sum())

display(x_non_mfe_df.describe().T.head(20))

(65, 372)


,5'UTR.Length,CDS.Length,3'UTR.Length,5'UTR.A-freq,5'UTR.U-freq,5'UTR.G-freq,5'UTR.C-freq,CDS.A-freq,CDS.U-freq,CDS.G-freq,...,CDS.dicodon_CUCAUA,CDS.dicodon_CUCCCG,CDS.dicodon_CUGAUA,CDS.dicodon_CUGCCG,CDS.dicodon_CUGCGA,CDS.dicodon_CUGCUG,CDS.dicodon_CUUCUG,CDS.dicodon_GUACCG,CDS.dicodon_GUACGA,CDS.dicodon_GUGCGA
0,86,393,149,197.674419,337.209302,174.418605,290.697674,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,86,393,148,197.674419,337.209302,174.418605,290.697674,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,89,393,148,202.247191,337.078652,179.775281,280.898876,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,195,312,216,251.282051,405.128205,143.589744,200.000000,272.435897,214.743590,314.102564,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,73,312,216,356.164384,315.068493,136.986301,191.780822,272.435897,214.743590,314.102564,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


shape: (65, 372)
missing values: 0
infinite values: 0
duplicated columns: 0


,count,mean,std,min,25%,50%,75%,max
5'UTR.Length,65.0,62.307692,34.017423,5.000000,44.000000,58.000000,78.000000,195.000000
CDS.Length,65.0,389.030769,202.720788,186.000000,312.000000,369.000000,432.000000,1425.000000
3'UTR.Length,65.0,185.830769,71.527217,11.000000,148.000000,166.000000,216.000000,414.000000
5'UTR.A-freq,65.0,295.096829,99.719133,125.000000,240.000000,281.250000,333.333333,600.000000
5'UTR.U-freq,65.0,277.747351,92.000735,0.000000,232.876712,300.000000,321.428571,440.677966
5'UTR.G-freq,65.0,192.255735,65.545943,57.142857,154.929577,186.440678,234.042553,428.571429
5'UTR.C-freq,65.0,234.900084,88.311725,0.000000,187.500000,225.352113,290.697674,416.666667
CDS.A-freq,65.0,289.661042,55.539720,179.487179,255.747126,279.898219,333.333333,386.206897
CDS.U-freq,65.0,225.058616,53.183031,96.774194,193.384224,219.966159,264.367816,314.553991
CDS.G-freq,65.0,268.307326,28.481474,208.791209,261.754386,270.114943,280.459770,327.586207


In [20]:
# ============================================
# MFE precompute wrapper
# ============================================

def get_mfe_file_path(seq_file):
    seq_file = Path(seq_file)
    return seq_file.with_name(
        seq_file.name.replace(".RNA.seq_data.tsv.gz", ".MFE.tsv.gz")
    )


def ensure_mfe_file(seq_file, temperature, n_workers=10, force=False):
    seq_file = Path(seq_file)
    mfe_file = get_mfe_file_path(seq_file)

    if mfe_file.exists() and not force:
        logger.info("MFE file already exists: %s", mfe_file)
        return mfe_file

    cmd = [
        sys.executable,
        "../scripts/precompute_mfe.py",
        str(seq_file),
        str(mfe_file),
        "--temperature",
        str(temperature),
        "-p",
        str(n_workers),
    ]

    logger.info("Running MFE script:")
    logger.info(" ".join(cmd))

    subprocess.run(cmd, check=True)

    return mfe_file

In [21]:
# ============================================
# Test MFE precompute wrapper: AT21
# ============================================

species = "AT21"

mfe_file = ensure_mfe_file(
    seq_file=species_config[species]["file"],
    temperature=species_config[species]["rna_temp"],
    n_workers=n_jobs,
    force=False,
)

print(mfe_file)
print(mfe_file.exists())

2026-06-14 18:38:00,368     INFO: Running MFE script:
2026-06-14 18:38:00,369     INFO: /home/ha-ibnu/miniconda3/envs/ibnu/bin/python ../scripts/precompute_mfe.py ../data/raw/AT21.PR_var.dev_data.RNA.seq_data.tsv.gz ../data/raw/AT21.PR_var.dev_data.MFE.tsv.gz --temperature 22 -p 10
2026-06-14 18:38:01,947     INFO: loading input: ../data/raw/AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:38:01,954     INFO: Input data |   name = AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:38:01,954     INFO: Input data |    md5 = 463f63c03bb16bfb05e1f6ea025947b8
2026-06-14 18:38:01,954     INFO: Input data | md5_gz = 7940f4f1aa886ba02c249422c074027e
2026-06-14 18:38:01,955     INFO: calculating MFE.5'UTR
2026-06-14 18:38:02,007     INFO: completed 65 / 65
2026-06-14 18:38:02,014     INFO: calculating MFE.CDS
2026-06-14 18:38:03,241     INFO: completed 65 / 65
2026-06-14 18:38:03,248     INFO: calculating MFE.3'UTR
2026-06-14 18:38:03,442     INFO: completed 65 / 65
2026-06-14 18:38:0

../data/raw/AT21.PR_var.dev_data.MFE.tsv.gz
True


2026-06-14 18:38:05,643     INFO: completed 65 / 65
2026-06-14 18:38:05,650     INFO: writing output: ../data/raw/AT21.PR_var.dev_data.MFE.tsv.gz
2026-06-14 18:38:05,653     INFO: done


In [22]:
# ============================================
# Load and check MFE features: AT21
# ============================================

mfe_df, mfe_metadata = sylib.fileio.load_df(str(mfe_file))

mfe_metadata.print_minimum_data(
    label=f"{species} MFE",
    logger=logger,
    logging_level="info",
)

print(mfe_df.shape)
display(mfe_df.head())

2026-06-14 18:38:31,511     INFO: AT21 MFE |   name = AT21.PR_var.dev_data.MFE.tsv.gz
2026-06-14 18:38:31,511     INFO: AT21 MFE |    md5 = 919a6c2b813f7270ab43f352253f0f39
2026-06-14 18:38:31,512     INFO: AT21 MFE | md5_gz = 1127b9e8e7b833e9e6a4005fb68cf16d


(65, 7)


,var_id,trans_id,gene_id,MFE.5'UTR,MFE.CDS,MFE.3'UTR,MFE.mRNA
0,AT1G07770.2.2409543.2408264,AT1G07770.2,AT1G07770,-28.57,-160.449997,-39.34,-242.389999
1,AT1G07770.2.2409543.2408265,AT1G07770.2,AT1G07770,-28.57,-160.449997,-39.34,-242.389999
2,AT1G07770.3.2409543.2408265,AT1G07770.3,AT1G07770,-30.15,-160.449997,-39.34,-243.250000
3,AT1G07820.1.2421940.2421219,AT1G07820.1,AT1G07820,-45.48,-122.290001,-57.73,-265.089996
4,AT1G07820.2.2421937.2421219,AT1G07820.2,AT1G07820,-10.29,-122.290001,-57.73,-217.289993


In [23]:
# ============================================
# Merge non-MFE and MFE features
# ============================================

mfe_feature_cols = [
    "MFE.5'UTR",
    "MFE.CDS",
    "MFE.3'UTR",
    "MFE.mRNA",
]

# safety check: row order should match original data
assert (seq_df["var_id"].values == mfe_df["var_id"].values).all()

x_feat_df = pd.concat(
    [
        x_non_mfe_df.reset_index(drop=True),
        mfe_df[mfe_feature_cols].reset_index(drop=True),
    ],
    axis=1,
)

print(x_feat_df.shape)
print("missing values:", x_feat_df.isna().sum().sum())
print("infinite values:", np.isinf(x_feat_df.to_numpy()).sum())
print("duplicated columns:", x_feat_df.columns.duplicated().sum())

display(x_feat_df.head())

(65, 376)
missing values: 0
infinite values: 0
duplicated columns: 0


,5'UTR.Length,CDS.Length,3'UTR.Length,5'UTR.A-freq,5'UTR.U-freq,5'UTR.G-freq,5'UTR.C-freq,CDS.A-freq,CDS.U-freq,CDS.G-freq,...,CDS.dicodon_CUGCGA,CDS.dicodon_CUGCUG,CDS.dicodon_CUUCUG,CDS.dicodon_GUACCG,CDS.dicodon_GUACGA,CDS.dicodon_GUGCGA,MFE.5'UTR,MFE.CDS,MFE.3'UTR,MFE.mRNA
0,86,393,149,197.674419,337.209302,174.418605,290.697674,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,-28.57,-160.449997,-39.34,-242.389999
1,86,393,148,197.674419,337.209302,174.418605,290.697674,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,-28.57,-160.449997,-39.34,-242.389999
2,89,393,148,202.247191,337.078652,179.775281,280.898876,264.631043,284.987277,272.264631,...,0.0,0.0,0.0,0.0,0.0,0.0,-30.15,-160.449997,-39.34,-243.250000
3,195,312,216,251.282051,405.128205,143.589744,200.000000,272.435897,214.743590,314.102564,...,0.0,0.0,0.0,0.0,0.0,0.0,-45.48,-122.290001,-57.73,-265.089996
4,73,312,216,356.164384,315.068493,136.986301,191.780822,272.435897,214.743590,314.102564,...,0.0,0.0,0.0,0.0,0.0,0.0,-10.29,-122.290001,-57.73,-217.289993


In [24]:
# ============================================
# Build complete feature matrix for one species
# ============================================

def build_species_feature_matrix(species, species_config, n_workers=10, force_mfe=False):
    logger.info("Building feature matrix for %s", species)

    seq_df, metadata = load_species_data(species, species_config)

    # non-MFE features
    x_non_mfe_df = build_non_mfe_features(seq_df)

    # MFE features
    mfe_file = ensure_mfe_file(
        seq_file=species_config[species]["file"],
        temperature=species_config[species]["rna_temp"],
        n_workers=n_workers,
        force=force_mfe,
    )

    mfe_df, mfe_metadata = sylib.fileio.load_df(str(mfe_file))

    # safety check
    assert (seq_df["var_id"].values == mfe_df["var_id"].values).all()

    mfe_feature_cols = [
        "MFE.5'UTR",
        "MFE.CDS",
        "MFE.3'UTR",
        "MFE.mRNA",
    ]

    x_feat_df = pd.concat(
        [
            x_non_mfe_df.reset_index(drop=True),
            mfe_df[mfe_feature_cols].reset_index(drop=True),
        ],
        axis=1,
    )

    # quality checks
    n_missing = x_feat_df.isna().sum().sum()
    n_infinite = np.isinf(x_feat_df.to_numpy()).sum()
    n_duplicated = x_feat_df.columns.duplicated().sum()

    logger.info(
        "%s features: shape=%s, missing=%s, infinite=%s, duplicated_cols=%s",
        species,
        x_feat_df.shape,
        n_missing,
        n_infinite,
        n_duplicated,
    )

    if n_missing > 0:
        raise ValueError(f"{species}: missing values found in feature matrix")
    if n_infinite > 0:
        raise ValueError(f"{species}: infinite values found in feature matrix")
    if n_duplicated > 0:
        raise ValueError(f"{species}: duplicated feature columns found")

    y_array = np.array(seq_df[score_col_name])

    return seq_df, x_feat_df, y_array

In [25]:
# ============================================
# Test feature matrix generation for all species
# ============================================

species_feature_data = {}

for species in species_config:
    seq_df_sp, x_feat_df_sp, y_array_sp = build_species_feature_matrix(
        species,
        species_config,
        n_workers=n_jobs,
        force_mfe=False,
    )

    species_feature_data[species] = {
        "seq_df": seq_df_sp,
        "x_feat_df": x_feat_df_sp,
        "y_array": y_array_sp,
    }

    print(species, seq_df_sp.shape, x_feat_df_sp.shape, y_array_sp.shape)

2026-06-14 18:45:21,731     INFO: Building feature matrix for AT21
2026-06-14 18:45:21,741     INFO: AT21 |   name = AT21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:21,742     INFO: AT21 |    md5 = 463f63c03bb16bfb05e1f6ea025947b8
2026-06-14 18:45:21,743     INFO: AT21 | md5_gz = 7940f4f1aa886ba02c249422c074027e
2026-06-14 18:45:21,826     INFO: MFE file already exists: ../data/raw/AT21.PR_var.dev_data.MFE.tsv.gz
2026-06-14 18:45:21,829     INFO: AT21 features: shape=(65, 376), missing=0, infinite=0, duplicated_cols=0
2026-06-14 18:45:21,829     INFO: Building feature matrix for NB21
2026-06-14 18:45:21,851     INFO: NB21 |   name = NB21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:21,852     INFO: NB21 |    md5 = cb81365e736f8802756f1a587f31c2db
2026-06-14 18:45:21,852     INFO: NB21 | md5_gz = 15627c4927492c9bddc2ef2a107d1986
2026-06-14 18:45:21,997     INFO: Running MFE script:
2026-06-14 18:45:21,998     INFO: /home/ha-ibnu/miniconda3/envs/ibnu/bin/python ../scrip

AT21 (65, 11) (65, 376) (65,)


2026-06-14 18:45:23,628     INFO: loading input: ../data/raw/NB21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:23,646     INFO: Input data |   name = NB21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:23,646     INFO: Input data |    md5 = cb81365e736f8802756f1a587f31c2db
2026-06-14 18:45:23,646     INFO: Input data | md5_gz = 15627c4927492c9bddc2ef2a107d1986
2026-06-14 18:45:23,647     INFO: calculating MFE.5'UTR
2026-06-14 18:45:23,718     INFO: completed 100 / 500
2026-06-14 18:45:23,741     INFO: completed 200 / 500
2026-06-14 18:45:23,767     INFO: completed 300 / 500
2026-06-14 18:45:23,789     INFO: completed 400 / 500
2026-06-14 18:45:23,867     INFO: completed 500 / 500
2026-06-14 18:45:23,874     INFO: calculating MFE.CDS
2026-06-14 18:45:25,479     INFO: completed 100 / 500
2026-06-14 18:45:27,035     INFO: completed 200 / 500
2026-06-14 18:45:28,863     INFO: completed 300 / 500
2026-06-14 18:45:30,210     INFO: completed 400 / 500
2026-06-14 18:45:31,922    

NB21 (500, 11) (500, 376) (500,)


2026-06-14 18:45:52,447     INFO: loading input: ../data/raw/OS21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:52,456     INFO: Input data |   name = OS21.PR_var.dev_data.RNA.seq_data.tsv.gz
2026-06-14 18:45:52,456     INFO: Input data |    md5 = 8d0fecf64cd0d7e31a3e43be59292070
2026-06-14 18:45:52,456     INFO: Input data | md5_gz = 7c64f7c0dd96839119b21df2563d9eec
2026-06-14 18:45:52,456     INFO: calculating MFE.5'UTR
2026-06-14 18:45:52,532     INFO: completed 100 / 117
2026-06-14 18:45:52,549     INFO: completed 117 / 117
2026-06-14 18:45:52,558     INFO: calculating MFE.CDS
2026-06-14 18:45:54,956     INFO: completed 100 / 117
2026-06-14 18:45:55,419     INFO: completed 117 / 117
2026-06-14 18:45:55,428     INFO: calculating MFE.3'UTR
2026-06-14 18:45:55,779     INFO: completed 100 / 117
2026-06-14 18:45:55,819     INFO: completed 117 / 117
2026-06-14 18:45:55,828     INFO: calculating MFE.mRNA
2026-06-14 18:46:01,193     INFO: completed 100 / 117
2026-06-14 18:46:01,976 

OS21 (117, 11) (117, 376) (117,)


In [26]:
# ============================================
# Feature matrix summary across species
# ============================================

summary_rows = []

for species, data in species_feature_data.items():
    x_feat_df = data["x_feat_df"]

    summary_rows.append({
        "species": species,
        "n_samples": x_feat_df.shape[0],
        "n_features": x_feat_df.shape[1],
        "n_missing": x_feat_df.isna().sum().sum(),
        "n_infinite": np.isinf(x_feat_df.to_numpy()).sum(),
        "n_duplicated_cols": x_feat_df.columns.duplicated().sum(),
        "n_constant_features": (x_feat_df.nunique() <= 1).sum(),
    })

feature_summary_df = pd.DataFrame(summary_rows)

display(feature_summary_df)

,species,n_samples,n_features,n_missing,n_infinite,n_duplicated_cols,n_constant_features
0,AT21,65,376,0,0,0,14
1,NB21,500,376,0,0,0,2
2,OS21,117,376,0,0,0,13


In [28]:
# ============================================
# Preprocessing: split, outlier removal, transform
# ============================================

xform_method = "yeo-johnson"
outlier_removal_method = "Z-score"
z_score_expected_n = 0.5
lof_contamination = "auto"

outlier_det_col_list = [
    "5'UTR.Length",
    "CDS.Length",
    "3'UTR.Length",
]

def detect_outliers(
    xformed_train_array,
    xformed_test_array,
    removal_method,
    z_score_expected_n=0.5,
    lof_contamination="auto",
):
    if removal_method is None:
        train_bool_array = np.ones(len(xformed_train_array), dtype=bool)
        test_bool_array = np.ones(len(xformed_test_array), dtype=bool)

    elif removal_method == "Z-score":
        min_thresh = stats.norm.ppf(
            (z_score_expected_n / 2) / len(xformed_train_array),
            loc=0,
            scale=1,
        )
        max_thresh = -min_thresh

        train_bool_array = (
            (xformed_train_array >= min_thresh)
            & (xformed_train_array <= max_thresh)
        )
        test_bool_array = (
            (xformed_test_array >= min_thresh)
            & (xformed_test_array <= max_thresh)
        )

    elif removal_method == "LOF":
        clf = LocalOutlierFactor(
            novelty=True,
            contamination=lof_contamination,
        ).fit(xformed_train_array.reshape(-1, 1))

        train_bool_array = clf.predict(xformed_train_array.reshape(-1, 1)) == 1
        test_bool_array = clf.predict(xformed_test_array.reshape(-1, 1)) == 1

    else:
        raise ValueError(f"Unknown outlier removal method: {removal_method}")

    return train_bool_array, test_bool_array

In [30]:
# ============================================
# Prepare train/test data for benchmarking
# ============================================

def prepare_train_test_data(
    seq_df,
    x_feat_df,
    y_array,
    group_col_name=None,
    test_data_ratio=0.2,
    random_state=0,
):
    # split
    if group_col_name is None:
        n_data = len(seq_df)
        n_test = int(round(n_data * test_data_ratio))

        train_idx_array, test_idx_array = model_selection.train_test_split(
            np.arange(n_data),
            random_state=random_state,
            test_size=n_test,
        )

        g_train, g_test = None, None

    else:
        g_sr = seq_df[group_col_name]
        gss = model_selection.GroupShuffleSplit(
            n_splits=1,
            test_size=test_data_ratio,
            random_state=random_state,
        )

        train_idx_array, test_idx_array = list(
            gss.split(np.arange(len(g_sr)), groups=g_sr)
        )[0]

        g_train = g_sr.iloc[train_idx_array].copy()
        g_test = g_sr.iloc[test_idx_array].copy()

    y_train_raw = y_array[train_idx_array].copy()
    y_test_raw = y_array[test_idx_array].copy()

    x_train_raw = x_feat_df.iloc[train_idx_array, :].copy()
    x_test_raw = x_feat_df.iloc[test_idx_array, :].copy()

    # y transform for outlier detection
    pre_y_train, y_xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
        y_train_raw,
        xform_method=xform_method,
        should_stdzn=True,
    )
    pre_y_test = sylib.utils.xform_array(y_test_raw, **y_xform_kwargs)

    train_bool_array, test_bool_array = detect_outliers(
        pre_y_train,
        pre_y_test,
        outlier_removal_method,
        z_score_expected_n=z_score_expected_n,
        lof_contamination=lof_contamination,
    )

    # outlier detection using selected X columns
    for col_name in outlier_det_col_list:
        if col_name not in x_train_raw.columns:
            continue

        if x_train_raw[col_name].var() == 0:
            continue

        xformed_train_array, xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
            x_train_raw[col_name],
            xform_method=xform_method,
            should_stdzn=True,
        )
        xformed_test_array = sylib.utils.xform_array(
            x_test_raw[col_name],
            **xform_kwargs,
        )

        temp_train_bool_array, temp_test_bool_array = detect_outliers(
            xformed_train_array,
            xformed_test_array,
            outlier_removal_method,
            z_score_expected_n=z_score_expected_n,
            lof_contamination=lof_contamination,
        )

        train_bool_array &= temp_train_bool_array
        test_bool_array &= temp_test_bool_array

    # remove outliers
    y_train_raw = y_train_raw[train_bool_array]
    y_test_raw = y_test_raw[test_bool_array]

    x_train = x_train_raw[train_bool_array].copy()
    x_test = x_test_raw[test_bool_array].copy()

    if group_col_name is not None:
        g_train = g_train.iloc[train_bool_array].copy()
        g_test = g_test.iloc[test_bool_array].copy()

    # final y transform after outlier removal
    y_train, y_xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
        y_train_raw,
        xform_method=xform_method,
        should_stdzn=True,
    )
    y_test = sylib.utils.xform_array(y_test_raw, **y_xform_kwargs)

    # transform X using training-derived parameters
    removed_features = []
    x_xform_param_dict = {}

    for col_name in list(x_train.columns):
        if x_train[col_name].var() == 0:
            del x_train[col_name]
            del x_test[col_name]
            x_xform_param_dict[col_name] = None
            removed_features.append(col_name)
            continue

        if len(set(x_train[col_name])) == 2:
            x_train[col_name], xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
                x_train[col_name],
                xform_method=None,
                should_stdzn=True,
            )
        else:
            x_train[col_name], xform_kwargs = sylib.utils.calc_xform_param_and_xform_array(
                x_train[col_name],
                xform_method=xform_method,
                should_stdzn=True,
            )

        x_test[col_name] = sylib.utils.xform_array(x_test[col_name], **xform_kwargs)
        x_xform_param_dict[col_name] = xform_kwargs

    info = {
        "n_before": len(seq_df),
        "n_train_before_outlier": len(x_train_raw),
        "n_test_before_outlier": len(x_test_raw),
        "n_train_after_outlier": len(x_train),
        "n_test_after_outlier": len(x_test),
        "n_features_before_filter": x_feat_df.shape[1],
        "n_features_after_filter": x_train.shape[1],
        "n_removed_train_outliers": len(x_train_raw) - len(x_train),
        "n_removed_test_outliers": len(x_test_raw) - len(x_test),
        "n_removed_features": len(removed_features),
        "removed_features": removed_features,
    }

    return {
        "x_train": x_train,
        "x_test": x_test,
        "y_train": y_train,
        "y_test": y_test,
        "g_train": g_train,
        "g_test": g_test,
        "info": info,
    }

In [31]:
# ============================================
# Prepare train/test data for all species
# ============================================

prepared_data = {}
prep_summary_rows = []

for species, data in species_feature_data.items():
    prepared = prepare_train_test_data(
        seq_df=data["seq_df"],
        x_feat_df=data["x_feat_df"],
        y_array=data["y_array"],
        group_col_name=group_col_name,
        test_data_ratio=test_data_ratio,
        random_state=random_state,
    )

    prepared_data[species] = prepared

    row = {"species": species}
    row.update(prepared["info"])
    prep_summary_rows.append(row)

prep_summary_df = pd.DataFrame(prep_summary_rows)

display(prep_summary_df)

,species,n_before,n_train_before_outlier,n_test_before_outlier,n_train_after_outlier,n_test_after_outlier,n_features_before_filter,n_features_after_filter,n_removed_train_outliers,n_removed_test_outliers,n_removed_features,removed_features
0,AT21,65,52,13,49,10,376,361,3,3,15,"[5'UTR.uORF-IO, CDS.dicodon_AUACGA, CDS.dicodo..."
1,NB21,500,400,100,395,100,376,373,5,0,3,"[CDS.dicodon_CGAAUA, CDS.dicodon_CGACGG, CDS.d..."
2,OS21,117,94,23,92,19,376,363,2,4,13,"[5'UTR.uORF-IO, CDS.dicodon_AGGCGA, CDS.dicodo..."


In [34]:
# ============================================
# Model evaluation helper
# ============================================

def evaluate_predictions(y_true, y_pred):
    pearson_r, _ = stats.pearsonr(y_true, y_pred)

    return {
        "R2": metrics.r2_score(y_true, y_pred),
        "Pearson_r": pearson_r,
        "RMSE": np.sqrt(metrics.mean_squared_error(y_true, y_pred)),
        "MAE": metrics.mean_absolute_error(y_true, y_pred),
    }

# ============================================
# Run one default model
# ============================================

def run_default_model(species, model_name, model, prepared, k_cv=10, n_jobs=10):
    x_train = prepared["x_train"]
    x_test = prepared["x_test"]
    y_train = prepared["y_train"]
    y_test = prepared["y_test"]
    g_train = prepared["g_train"]

    if g_train is None:
        cv = model_selection.KFold(
            n_splits=k_cv,
            shuffle=True,
            random_state=random_state,
        )
        cv_groups = None
    else:
        cv = model_selection.GroupKFold(n_splits=k_cv)
        cv_groups = g_train

    cv_scores = model_selection.cross_validate(
        model,
        x_train,
        y_train,
        cv=cv,
        groups=cv_groups,
        scoring="r2",
        n_jobs=n_jobs,
        return_train_score=False,
    )

    model.fit(x_train, y_train)

    pred_train = model.predict(x_train)
    pred_test = model.predict(x_test)

    train_metrics = evaluate_predictions(y_train, pred_train)
    test_metrics = evaluate_predictions(y_test, pred_test)

    return {
        "species": species,
        "model": model_name,
        "setting": "default",
        "CV_R2": cv_scores["test_score"].mean(),
        "CV_R2_sd": cv_scores["test_score"].std(),
        "Train_R2": train_metrics["R2"],
        "Test_R2": test_metrics["R2"],
        "Train_Pearson_r": train_metrics["Pearson_r"],
        "Test_Pearson_r": test_metrics["Pearson_r"],
        "Train_RMSE": train_metrics["RMSE"],
        "Test_RMSE": test_metrics["RMSE"],
        "Train_MAE": train_metrics["MAE"],
        "Test_MAE": test_metrics["MAE"],
    }

In [35]:
# ============================================
# Test benchmark pipeline with Ridge
# ============================================

ridge_result = run_default_model(
    species="AT21",
    model_name="Ridge",
    model=default_models["Ridge"],
    prepared=prepared_data["AT21"],
    k_cv=k_cv,
    n_jobs=n_jobs,
)

pd.DataFrame([ridge_result])

,species,model,setting,CV_R2,CV_R2_sd,Train_R2,Test_R2,Train_Pearson_r,Test_Pearson_r,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE
0,AT21,Ridge,default,-1.152653,2.415388,0.977061,-2.029304,0.9886,0.348907,0.151458,1.538329,0.083581,1.298798


In [36]:
# ============================================
# Test all default models on AT21
# ============================================

default_results_at21 = []

for model_name, model in get_default_models(random_state=random_state, n_jobs=n_jobs).items():
    logger.info("Running AT21 default model: %s", model_name)

    result = run_default_model(
        species="AT21",
        model_name=model_name,
        model=model,
        prepared=prepared_data["AT21"],
        k_cv=k_cv,
        n_jobs=n_jobs,
    )

    default_results_at21.append(result)

default_results_at21_df = pd.DataFrame(default_results_at21)
display(default_results_at21_df)

2026-06-14 18:58:44,159     INFO: Running AT21 default model: PLS
2026-06-14 18:58:44,229     INFO: Running AT21 default model: Ridge
2026-06-14 18:58:44,297     INFO: Running AT21 default model: Lasso
2026-06-14 18:58:44,590     INFO: Running AT21 default model: MLP
2026-06-14 18:58:44,794     INFO: Running AT21 default model: DecisionTree
2026-06-14 18:58:44,892     INFO: Running AT21 default model: RandomForest
2026-06-14 18:58:45,649     INFO: Running AT21 default model: XGBoost
2026-06-14 18:58:47,507     INFO: Running AT21 default model: LightGBM


,species,model,setting,CV_R2,CV_R2_sd,Train_R2,Test_R2,Train_Pearson_r,Test_Pearson_r,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE
0,AT21,PLS,default,-1.414040,3.089140,0.883737,-1.817397,0.940073,0.244806,0.340974,1.483549,0.256798,1.253005
1,AT21,Ridge,default,-1.152653,2.415388,0.977061,-2.029304,0.988600,0.348907,0.151458,1.538329,0.083581,1.298798
2,AT21,Lasso,default,-2.457676,3.407636,0.992570,-6.285629,0.996343,0.506825,0.086196,2.385676,0.046362,1.931711
3,AT21,MLP,default,-1.720144,3.757800,0.974632,-1.915271,0.987276,0.329234,0.159274,1.509098,0.077507,1.271678
4,AT21,DecisionTree,default,-1.507271,3.431074,1.000000,-2.072319,1.000000,0.190852,0.000000,1.549213,0.000000,1.419139
5,AT21,RandomForest,default,-0.521123,1.405545,0.902887,-0.740222,0.973300,0.189291,0.311630,1.165951,0.241847,0.979152
6,AT21,XGBoost,default,-1.241754,2.974793,1.000000,-1.791636,1.000000,0.224775,0.000706,1.476751,0.000573,1.300339
7,AT21,LightGBM,default,-1.036466,2.494456,0.862917,-0.968974,0.939055,0.175688,0.370248,1.240218,0.294473,1.014708


In [37]:
# ============================================
# Run default benchmark: all species × all models
# ============================================

default_results = []

for species in species_config:
    for model_name, model in get_default_models(random_state=random_state, n_jobs=n_jobs).items():
        logger.info("Running default model: species=%s, model=%s", species, model_name)

        result = run_default_model(
            species=species,
            model_name=model_name,
            model=model,
            prepared=prepared_data[species],
            k_cv=k_cv,
            n_jobs=n_jobs,
        )

        default_results.append(result)

default_results_df = pd.DataFrame(default_results)

display(default_results_df)

default_results_df.sort_values(["species", "Test_R2"], ascending=[True, False])

2026-06-14 18:59:20,093     INFO: Running default model: species=AT21, model=PLS
2026-06-14 18:59:20,161     INFO: Running default model: species=AT21, model=Ridge
2026-06-14 18:59:20,228     INFO: Running default model: species=AT21, model=Lasso
2026-06-14 18:59:20,520     INFO: Running default model: species=AT21, model=MLP
2026-06-14 18:59:20,709     INFO: Running default model: species=AT21, model=DecisionTree
2026-06-14 18:59:20,778     INFO: Running default model: species=AT21, model=RandomForest
2026-06-14 18:59:21,556     INFO: Running default model: species=AT21, model=XGBoost
2026-06-14 18:59:22,379     INFO: Running default model: species=AT21, model=LightGBM
2026-06-14 18:59:23,116     INFO: Running default model: species=NB21, model=PLS
2026-06-14 18:59:23,191     INFO: Running default model: species=NB21, model=Ridge
2026-06-14 18:59:23,277     INFO: Running default model: species=NB21, model=Lasso
2026-06-14 18:59:23,975     INFO: Running default model: species=NB21, mod

,species,model,setting,CV_R2,CV_R2_sd,Train_R2,Test_R2,Train_Pearson_r,Test_Pearson_r,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE
0,AT21,PLS,default,-1.414040,3.089140,0.883737,-1.817397,0.940073,0.244806,0.340974,1.483549,0.256798,1.253005
1,AT21,Ridge,default,-1.152653,2.415388,0.977061,-2.029304,0.988600,0.348907,0.151458,1.538329,0.083581,1.298798
2,AT21,Lasso,default,-2.457676,3.407636,0.992570,-6.285629,0.996343,0.506825,0.086196,2.385676,0.046362,1.931711
3,AT21,MLP,default,-1.720144,3.757800,0.974632,-1.915271,0.987276,0.329234,0.159274,1.509098,0.077507,1.271678
4,AT21,DecisionTree,default,-1.507271,3.431074,1.000000,-2.072319,1.000000,0.190852,0.000000,1.549213,0.000000,1.419139
5,AT21,RandomForest,default,-0.521123,1.405545,0.902887,-0.740222,0.973300,0.189291,0.311630,1.165951,0.241847,0.979152
6,AT21,XGBoost,default,-1.241754,2.974793,1.000000,-1.791636,1.000000,0.224775,0.000706,1.476751,0.000573,1.300339
7,AT21,LightGBM,default,-1.036466,2.494456,0.862917,-0.968974,0.939055,0.175688,0.370248,1.240218,0.294473,1.014708
8,NB21,PLS,default,-0.173707,0.173230,0.628287,-0.573997,0.792646,0.121554,0.609683,1.309977,0.481215,1.040054
9,NB21,Ridge,default,-1.799306,0.682347,0.908803,-3.128204,0.953928,-0.020640,0.301988,2.121496,0.237069,1.545588


,species,model,setting,CV_R2,CV_R2_sd,Train_R2,Test_R2,Train_Pearson_r,Test_Pearson_r,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE
5,AT21,RandomForest,default,-0.521123,1.405545,0.902887,-0.740222,0.973300,0.189291,0.311630,1.165951,0.241847,0.979152
7,AT21,LightGBM,default,-1.036466,2.494456,0.862917,-0.968974,0.939055,0.175688,0.370248,1.240218,0.294473,1.014708
6,AT21,XGBoost,default,-1.241754,2.974793,1.000000,-1.791636,1.000000,0.224775,0.000706,1.476751,0.000573,1.300339
0,AT21,PLS,default,-1.414040,3.089140,0.883737,-1.817397,0.940073,0.244806,0.340974,1.483549,0.256798,1.253005
3,AT21,MLP,default,-1.720144,3.757800,0.974632,-1.915271,0.987276,0.329234,0.159274,1.509098,0.077507,1.271678
1,AT21,Ridge,default,-1.152653,2.415388,0.977061,-2.029304,0.988600,0.348907,0.151458,1.538329,0.083581,1.298798
4,AT21,DecisionTree,default,-1.507271,3.431074,1.000000,-2.072319,1.000000,0.190852,0.000000,1.549213,0.000000,1.419139
2,AT21,Lasso,default,-2.457676,3.407636,0.992570,-6.285629,0.996343,0.506825,0.086196,2.385676,0.046362,1.931711
13,NB21,RandomForest,default,0.193518,0.076846,0.893012,0.129508,0.978858,0.377571,0.327091,0.974191,0.262416,0.738554
14,NB21,XGBoost,default,0.109014,0.116565,0.999995,0.063124,0.999998,0.319809,0.002250,1.010655,0.001587,0.768418


In [45]:
# ============================================
# Tuning helpers
# ============================================

def get_cv(prepared, k_cv=10, random_state=0):
    g_train = prepared["g_train"]
    n_train = len(prepared["y_train"])

    # avoid invalid CV when data are small after outlier removal
    effective_k_cv = min(k_cv, n_train)

    if effective_k_cv < 2:
        raise ValueError(f"Not enough training samples for CV: n_train={n_train}")

    if g_train is None:
        cv = model_selection.KFold(
            n_splits=effective_k_cv,
            shuffle=True,
            random_state=random_state,
        )
        cv_groups = None
    else:
        random.seed(random_state)
        np.random.seed(random_state)

        n_groups = len(set(g_train))
        effective_k_cv = min(effective_k_cv, n_groups)

        if effective_k_cv < 2:
            raise ValueError(f"Not enough groups for GroupKFold: n_groups={n_groups}")

        cv = model_selection.GroupKFold(n_splits=effective_k_cv)
        cv_groups = g_train

    return cv, cv_groups


def collect_optuna_trials(study):
    rows = []

    for trial in study.get_trials():
        row = {
            "trial": trial.number,
            "value": trial.value,
            "state": str(trial.state),
        }
        row.update(trial.params)
        rows.append(row)

    trial_df = pd.DataFrame(rows)

    if len(trial_df) > 0 and "value" in trial_df.columns:
        trial_df = trial_df.sort_values("value", ascending=False)

    return trial_df


def get_tuned_model(model_name, trial, prepared=None, random_state=0, n_jobs=10):
    if prepared is not None:
        n_train = prepared["x_train"].shape[0]
        n_features = prepared["x_train"].shape[1]
    else:
        n_train = 100
        n_features = 100

    if model_name == "PLS":
        max_components = max(2, min(30, n_train - 1, n_features))

        params = {
            "n_components": trial.suggest_int("n_components", 2, max_components),
        }

        model = PLSRegression(**params)

    elif model_name == "Ridge":
        params = {
            "alpha": trial.suggest_float("alpha", 1e-4, 1e4, log=True),
        }

        model = Ridge(**params)

    elif model_name == "Lasso":
        params = {
            "alpha": trial.suggest_float("alpha", 1e-5, 1e1, log=True),
        }

        model = Lasso(
            max_iter=20000,
            random_state=random_state,
            **params,
        )

    elif model_name == "MLP":
        hidden_layer_choice = trial.suggest_categorical(
            "hidden_layer_sizes",
            [
                "64",
                "128",
                "256",
                "128_64",
                "256_128",
            ],
        )

        hidden_layer_map = {
            "64": (64,),
            "128": (128,),
            "256": (256,),
            "128_64": (128, 64),
            "256_128": (256, 128),
        }

        params = {
            "hidden_layer_sizes": hidden_layer_map[hidden_layer_choice],
            "alpha": trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
            "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
        }

        model = MLPRegressor(
            activation="relu",
            solver="adam",
            max_iter=1000,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=20,
            random_state=random_state,
            **params,
        )

    elif model_name == "DecisionTree":
        params = {
            "max_depth": trial.suggest_int("max_depth", 2, 30),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        }

        model = DecisionTreeRegressor(
            random_state=random_state,
            **params,
        )

    elif model_name == "RandomForest":
        params = {
            "max_depth": trial.suggest_int("max_depth", 3, 30),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "max_features": trial.suggest_float("max_features", 0.2, 1.0, step=0.1),
        }

        model = RandomForestRegressor(
            n_estimators=300,
            random_state=random_state,
            n_jobs=1,
            **params,
        )

    elif model_name == "XGBoost":
        params = {
            "max_depth": trial.suggest_int("max_depth", 2, 10),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0, step=0.1),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0, step=0.1),
            "min_child_weight": trial.suggest_float("min_child_weight", 1e-2, 10.0, log=True),
        }

        model = XGBRegressor(
            objective="reg:squarederror",
            n_estimators=300,
            learning_rate=0.05,
            random_state=random_state,
            n_jobs=1,
            **params,
        )

    elif model_name == "LightGBM":
        params = {
            "max_depth": trial.suggest_int("max_depth", 3, 20),
            "min_child_samples": trial.suggest_int("min_child_samples", 2, 30),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.2, 1.0, step=0.1),
            "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        }

        model = LGBMRegressor(
            objective="regression",
            n_estimators=100,
            learning_rate=0.05,
            random_state=random_state,
            n_jobs=1,
            verbosity=-1,
            **params,
        )

    else:
        raise ValueError(f"Unknown model name: {model_name}")

    return model


# ============================================
# Generic Optuna tuner
# ============================================

def tune_model(
    model_name,
    prepared,
    n_trials=20,
    k_cv=10,
    random_state=0,
    n_jobs=10,
):
    x_train = prepared["x_train"]
    y_train = prepared["y_train"]

    cv, cv_groups = get_cv(
        prepared,
        k_cv=k_cv,
        random_state=random_state,
    )

    def objective(trial):
        model = get_tuned_model(
            model_name=model_name,
            trial=trial,
            prepared=prepared,
            random_state=random_state,
            n_jobs=n_jobs,
        )

        try:
            scores = model_selection.cross_validate(
                model,
                x_train,
                y_train,
                cv=cv,
                groups=cv_groups,
                n_jobs=n_jobs,
                scoring="r2",
                return_train_score=False,
                error_score=np.nan,
            )

            mean_score = np.nanmean(scores["test_score"])

            if np.isnan(mean_score):
                return -1e9

            return mean_score

        except Exception as e:
            logger.warning(
                "Trial failed: model=%s, error=%s",
                model_name,
                e,
            )
            return -1e9

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=random_state),
    )

    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=True,
    )

    best_params = study.best_params
    best_cv_r2 = study.best_value
    trial_df = collect_optuna_trials(study)

    return {
        "study": study,
        "best_params": best_params,
        "best_cv_r2": best_cv_r2,
        "trial_df": trial_df,
    }

In [46]:
# ============================================
# Run one tuned model
# ============================================

def run_tuned_model(
    species,
    model_name,
    prepared,
    n_trials=20,
    k_cv=10,
    random_state=0,
    n_jobs=10,
):
    tuning = tune_model(
        model_name=model_name,
        prepared=prepared,
        n_trials=n_trials,
        k_cv=k_cv,
        random_state=random_state,
        n_jobs=n_jobs,
    )

    best_trial = tuning["study"].best_trial

    model = get_tuned_model(
        model_name=model_name,
        trial=best_trial,
        prepared=prepared,
        random_state=random_state,
        n_jobs=n_jobs,
    )

    x_train = prepared["x_train"]
    x_test = prepared["x_test"]
    y_train = prepared["y_train"]
    y_test = prepared["y_test"]

    model.fit(x_train, y_train)

    pred_train = model.predict(x_train)
    pred_test = model.predict(x_test)

    train_metrics = evaluate_predictions(y_train, pred_train)
    test_metrics = evaluate_predictions(y_test, pred_test)

    return {
        "species": species,
        "model": model_name,
        "setting": "tuned",
        "CV_R2": tuning["best_cv_r2"],
        "CV_R2_sd": np.nan,
        "Train_R2": train_metrics["R2"],
        "Test_R2": test_metrics["R2"],
        "Train_Pearson_r": train_metrics["Pearson_r"],
        "Test_Pearson_r": test_metrics["Pearson_r"],
        "Train_RMSE": train_metrics["RMSE"],
        "Test_RMSE": test_metrics["RMSE"],
        "Train_MAE": train_metrics["MAE"],
        "Test_MAE": test_metrics["MAE"],
        "best_params": tuning["best_params"],
        "trial_df": tuning["trial_df"],
    }

In [47]:
# ============================================
# Run tuned benchmark: all species × all models
# ============================================

tuned_results = []

for species in species_config:
    for model_name in default_models.keys():
        logger.info("Running tuned model: species=%s, model=%s", species, model_name)

        result = run_tuned_model(
            species=species,
            model_name=model_name,
            prepared=prepared_data[species],
            n_trials=n_trials,
            k_cv=k_cv,
            random_state=random_state,
            n_jobs=n_jobs,
        )

        tuned_results.append(result)

tuned_results_df = pd.DataFrame([
    {k: v for k, v in row.items() if k != "trial_df"}
    for row in tuned_results
])

display(tuned_results_df)

2026-06-14 19:59:42,921     INFO: Running tuned model: species=AT21, model=PLS
Best trial: 11. Best value: -0.986804: 100%|██████████| 20/20 [00:03<00:00,  5.42it/s]
2026-06-14 19:59:46,627     INFO: Running tuned model: species=AT21, model=Ridge
Best trial: 17. Best value: -0.439742: 100%|██████████| 20/20 [00:01<00:00, 15.67it/s]
2026-06-14 19:59:47,917     INFO: Running tuned model: species=AT21, model=Lasso
Best trial: 7. Best value: -0.594046: 100%|██████████| 20/20 [00:01<00:00, 10.95it/s]
2026-06-14 19:59:49,754     INFO: Running tuned model: species=AT21, model=MLP
Best trial: 15. Best value: -1.25883: 100%|██████████| 20/20 [00:02<00:00,  7.03it/s]
2026-06-14 19:59:52,627     INFO: Running tuned model: species=AT21, model=DecisionTree
Best trial: 0. Best value: -0.48549: 100%|██████████| 20/20 [00:01<00:00, 14.39it/s]
2026-06-14 19:59:54,027     INFO: Running tuned model: species=AT21, model=RandomForest
Best trial: 16. Best value: -0.399847: 100%|██████████| 20/20 [00:03<00:0

,species,model,setting,CV_R2,CV_R2_sd,Train_R2,Test_R2,Train_Pearson_r,Test_Pearson_r,Train_RMSE,Test_RMSE,Train_MAE,Test_MAE,best_params
0,AT21,PLS,tuned,-0.986804,NaN,0.940978,-1.850027,0.970040,0.242921,0.242944,1.492115,0.177756,1.278609,{'n_components': 10}
1,AT21,Ridge,tuned,-0.439742,NaN,0.530323,-0.117247,0.850459,0.223371,0.685330,0.934227,0.535750,0.772481,{'alpha': 863.8487008166876}
2,AT21,Lasso,tuned,-0.594046,NaN,0.000000,-0.029224,NaN,NaN,1.000000,0.896670,0.804289,0.754274,{'alpha': 2.2420123713724407}
3,AT21,MLP,tuned,-1.258825,NaN,-0.030504,0.144420,0.285938,0.568876,1.015137,0.817538,0.779241,0.680972,"{'hidden_layer_sizes': '128', 'alpha': 0.00786..."
4,AT21,DecisionTree,tuned,-0.485490,NaN,0.491878,-0.285915,0.701340,0.141660,0.712827,1.002268,0.522478,0.840279,"{'max_depth': 17, 'min_samples_leaf': 15, 'min..."
5,AT21,RandomForest,tuned,-0.399847,NaN,0.610629,-0.416379,0.850067,0.009932,0.623996,1.051883,0.462932,0.902308,"{'max_depth': 9, 'min_samples_leaf': 7, 'max_f..."
6,AT21,XGBoost,tuned,-0.945967,NaN,0.999903,-1.685617,0.999961,0.241151,0.009868,1.448438,0.006351,1.250546,"{'max_depth': 8, 'subsample': 0.6, 'colsample_..."
7,AT21,LightGBM,tuned,-0.492065,NaN,0.727632,-0.562528,0.884717,0.146431,0.521888,1.104821,0.400027,0.892983,"{'max_depth': 11, 'min_child_samples': 18, 'fe..."
8,NB21,PLS,tuned,0.116307,NaN,0.433305,-0.058966,0.658259,0.258255,0.752792,1.074491,0.596768,0.835602,{'n_components': 2}
9,NB21,Ridge,tuned,0.172228,NaN,0.407675,0.038124,0.692581,0.252300,0.769626,1.024050,0.620787,0.781822,{'alpha': 1362.021635265945}
